# Pseudolabeling Experiment (v2) — Drive-cached, resumable

**Goal:** Improve the Swin-Base baseline (0.9363 AUC) using pseudolabeling, the technique used by the SIIM-ISIC 2nd place Kaggle solution.

## What changed in v2

1. **Drive caching at every expensive step.** If the session dies mid-way, re-running this notebook resumes from the latest saved artefact rather than restarting from scratch.
2. **Corrected class indexing.** In the original Swin-Base training, `target=1` corresponds to *malignant* (melanoma) and the model was trained with the CSV's binary `target` as the class index directly. This means output index **1** is the melanoma channel, not index 0. The variable named `MEL_IDX = 0` in `train.py` is misleadingly named — see the discussion in Cell 9.
3. **Per-epoch student checkpoints.** Resume training mid-run instead of redoing finished epochs.
4. **Idempotent cells.** Re-running any cell is safe — cached results are reused.

## What is pseudolabeling?

Use the trained model to predict labels for the unlabelled test set, treat the high-confidence predictions as if they were real labels ("pseudo-labels"), and retrain on the combined pool.

Even noisy labels carry signal. Confident predictions are usually correct, and they add useful gradient information — especially for the minority (melanoma) class which is severely under-represented in real labelled data.

## Pipeline

1. **Predict** soft probabilities on the test set with the teacher (with TTA). → Cached in Drive.
2. **Filter** by confidence and inspect class distribution.
3. **Build** the combined training set (real + pseudo, with optional minority upsampling).
4. **Retrain** Swin-Base from scratch using a mixed hard+soft loss. → Per-epoch checkpoints in Drive.
5. **Evaluate** on the held-out validation fold (real labels only).

## Honest expectations

- Likely gain: +0.005 to +0.020 AUC
- Could also hurt (confirmation bias) — we'll monitor the pseudolabel class distribution to catch this early
- This is post-submission learning, not part of the original report

## Cell 1 — Mount Drive, Clone Repo, Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
%cd /content/

if not os.path.exists('/content/MelanomaClassificationAML'):
    !git clone https://github.com/PalsRoy/MelanomaClassificationAML.git
else:
    print('Repo already cloned, skipping')

!pip install -q timm albumentations

## Cell 2 — Unzip Data (idempotent)

In [ ]:
DRIVE_DATA = '/content/drive/MyDrive/Colab Notebooks/siim-isic-melanoma-classification'

!mkdir -p /content/data

if not os.path.exists('/content/data/jpeg/train') or len(os.listdir('/content/data/jpeg/train')) < 33000:
    print('Unzipping JPEG data (~10-15 min)...')
    !unzip -q -o '{DRIVE_DATA}/jpeg.zip' -d /content/data/
else:
    print('JPEG already extracted')

!cp '{DRIVE_DATA}/train.csv' /content/data/
!cp '{DRIVE_DATA}/test.csv' /content/data/

n_train = len(os.listdir('/content/data/jpeg/train'))
n_test = len(os.listdir('/content/data/jpeg/test'))
print(f'Train images: {n_train} / 33126')
print(f'Test images:  {n_test} / 10982')

## Cell 3 — Load Project Modules

In [ ]:
import importlib.util
import sys

REPO = '/content/MelanomaClassificationAML'

def load_module(name, filepath):
    spec = importlib.util.spec_from_file_location(name, filepath)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

config_module  = load_module('config',  f'{REPO}/config.py')
dataset_module = load_module('dataset', f'{REPO}/dataset.py')
models_module  = load_module('models',  f'{REPO}/models.py')
train_module   = load_module('train',   f'{REPO}/train.py')

CFG                  = config_module.CFG
MelanomaDataset      = dataset_module.MelanomaDataset
get_transforms       = dataset_module.get_transforms
build_model          = models_module.build_model
make_amp_components  = train_module.make_amp_components

print('All modules loaded.')

## Cell 4 — Pseudolabeling Configuration

In [ ]:
# === EXPERIMENT IDENTITY ===
EXPERIMENT_NAME = 'exp6_swin_pseudolabel'

# === MATCH THE TEACHER ARCHITECTURE ===
CFG.model_name = 'swin_base_patch4_window7_224'
CFG.image_size = 224
CFG.batch_size = 32                 # match the original Swin-Base run
CFG.DATA_DIR    = '/content/data'
CFG.JPEG_DIR    = '/content/data/jpeg'
CFG.num_workers = 4
CFG.use_amp     = True
CFG.n_epochs    = 5                 # match the original 5-epoch comparison budget
CFG.fold        = 0

# === CRITICAL: melanoma is class 1 (matches your training convention) ===
# Your train.csv encodes target=1 for malignant, target=0 for benign.
# Your model trained on raw targets as class indices, so output index 1 = melanoma.
# (Note: train.py's MEL_IDX=0 is misnamed; AUC was numerically correct because
# ROC-AUC is symmetric under label inversion in the binary case.)
MELANOMA_IDX = 1

# === TEACHER CHECKPOINT (your best Swin-Base) ===
TEACHER_CHECKPOINT = '/content/drive/MyDrive/melanoma_results/weights/exp4_swin_base_fold0_best.pth'

# === PSEUDOLABEL HYPERPARAMETERS ===
CONFIDENCE_THRESHOLD     = 0.80     # Keep test images where max class probability >= this
USE_SOFT_LABELS          = True     # Soft targets preserve uncertainty
PSEUDO_MELANOMA_UPSAMPLE = 7        # Kaggle 2nd place used 7×; set 1 to disable
USE_TTA_FOR_PSEUDOLABELS = True
TTA_PASSES               = 4

# === CACHE LOCATIONS (Drive) ===
CACHE_ROOT    = f'/content/drive/MyDrive/melanoma_results/pseudolabel/{EXPERIMENT_NAME}'
WEIGHTS_DIR   = '/content/drive/MyDrive/melanoma_results/weights'
RESULTS_DIR   = '/content/drive/MyDrive/melanoma_results/results'
os.makedirs(CACHE_ROOT,  exist_ok=True)
os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

TEST_PROBS_CACHE   = f'{CACHE_ROOT}/test_probs.npy'
PSEUDO_DF_CACHE    = f'{CACHE_ROOT}/df_pseudo.pkl'
COMBINED_DF_CACHE  = f'{CACHE_ROOT}/df_combined.pkl'
TRAIN_STATE_CACHE  = f'{CACHE_ROOT}/train_state.pt'   # for mid-training resume

print(f'Experiment:      {EXPERIMENT_NAME}')
print(f'Teacher model:   {TEACHER_CHECKPOINT}')
print(f'Melanoma class:  {MELANOMA_IDX}')
print(f'Conf threshold:  {CONFIDENCE_THRESHOLD}')
print(f'Soft labels:     {USE_SOFT_LABELS}')
print(f'Mel. upsample:   {PSEUDO_MELANOMA_UPSAMPLE}×')
print(f'TTA passes:      {TTA_PASSES if USE_TTA_FOR_PSEUDOLABELS else "disabled"}')
print(f'Cache root:      {CACHE_ROOT}')
print(f'Device:          {CFG.device}')

## Cell 5 — Prepare Folds (always recomputed; cheap)

Same fold split as the original Swin-Base run so the validation set is identical.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

df_train = pd.read_csv(os.path.join(CFG.DATA_DIR, 'train.csv'))
df_train['filepath'] = df_train['image_name'].apply(
    lambda x: os.path.join(CFG.JPEG_DIR, 'train', f'{x}.jpg')
)

sgkf = StratifiedGroupKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
df_train['fold'] = -1
for fold_idx, (_, val_idx) in enumerate(
    sgkf.split(df_train, df_train['target'], df_train['patient_id'])
):
    df_train.loc[val_idx, 'fold'] = fold_idx

df_val = df_train[df_train['fold'] == CFG.fold].reset_index(drop=True)
df_trn = df_train[df_train['fold'] != CFG.fold].reset_index(drop=True)

print(f'Train: {len(df_trn)} | Val: {len(df_val)}')
print(f'Melanoma (target==1) in train: {(df_trn["target"]==1).sum()} ({(df_trn["target"]==1).mean()*100:.2f}%)')
print(f'Melanoma (target==1) in val:   {(df_val["target"]==1).sum()} ({(df_val["target"]==1).mean()*100:.2f}%)')

# Load test set
df_test = pd.read_csv(os.path.join(CFG.DATA_DIR, 'test.csv'))
df_test['filepath'] = df_test['image_name'].apply(
    lambda x: os.path.join(CFG.JPEG_DIR, 'test', f'{x}.jpg')
)
print(f'Test:  {len(df_test)}')

## Cell 6 — Load Teacher Model (only if needed for pseudolabel generation)

If test_probs is already cached, we skip loading the teacher to save time and memory.

In [ ]:
import torch

TEACHER_LOADED = False
teacher = None

if os.path.exists(TEST_PROBS_CACHE):
    print(f'✓ Test predictions already cached at {TEST_PROBS_CACHE}')
    print(f'  Skipping teacher load. To regenerate, delete the cache file.')
else:
    print('No cached predictions found — loading teacher to generate them...')
    teacher = build_model(
        model_name=CFG.model_name,
        out_dim=9,
        pretrained=False,
        drop_rate=0.5,
    ).to(CFG.device)
    
    checkpoint = torch.load(TEACHER_CHECKPOINT, map_location=CFG.device, weights_only=False)
    teacher.load_state_dict(checkpoint['model_state_dict'])
    teacher.eval()
    TEACHER_LOADED = True
    
    print(f'Teacher loaded from epoch {checkpoint["epoch"]}')
    print(f'Teacher val AUC at save time: {checkpoint["best_auc"]:.4f}')

## Cell 7 — Generate Pseudolabels (CACHED)

**This is the expensive step (~15-20 min with TTA).** Results are saved to Drive immediately on completion. If the cache exists, this cell is essentially a no-op.

In [ ]:
from torch.utils.data import DataLoader
from tqdm import tqdm

@torch.no_grad()
def predict_test_set(model, df, transform, device, n_tta=1, batch_size=64):
    """Generate soft predictions over a dataset, optionally with TTA."""
    accumulated = None
    for tta_pass in range(n_tta):
        # Re-create the dataset/loader each pass so the random transforms differ
        ds = MelanomaDataset(df.assign(target=0), transform=transform)  # dummy target
        loader = DataLoader(
            ds, batch_size=batch_size, shuffle=False,
            num_workers=CFG.num_workers, pin_memory=True,
        )
        
        pass_probs = []
        desc = f'  TTA pass {tta_pass+1}/{n_tta}' if n_tta > 1 else '  Predict'
        for images, _ in tqdm(loader, desc=desc, leave=False):
            images = images.to(device, non_blocking=True)
            logits = model(images)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            pass_probs.append(probs)
        pass_probs = np.concatenate(pass_probs, axis=0)
        
        if accumulated is None:
            accumulated = pass_probs
        else:
            accumulated += pass_probs
    
    return accumulated / n_tta


if os.path.exists(TEST_PROBS_CACHE):
    print(f'✓ Loading cached predictions from {TEST_PROBS_CACHE}')
    test_probs = np.load(TEST_PROBS_CACHE)
    print(f'  Loaded: shape={test_probs.shape}')
else:
    assert TEACHER_LOADED, 'Teacher must be loaded if cache is absent. Re-run Cell 6.'
    
    if USE_TTA_FOR_PSEUDOLABELS:
        print(f'Generating pseudolabels with {TTA_PASSES}-pass TTA...')
        transform = get_transforms(CFG.image_size, 'train')
        test_probs = predict_test_set(
            teacher, df_test, transform, CFG.device,
            n_tta=TTA_PASSES, batch_size=CFG.batch_size * 2,
        )
    else:
        print('Generating pseudolabels (single pass, no TTA)...')
        transform = get_transforms(CFG.image_size, 'val')
        test_probs = predict_test_set(
            teacher, df_test, transform, CFG.device,
            n_tta=1, batch_size=CFG.batch_size * 2,
        )
    
    np.save(TEST_PROBS_CACHE, test_probs)
    print(f'✓ Saved predictions to {TEST_PROBS_CACHE}')

# Free teacher memory if we don't need it again
if teacher is not None:
    del teacher
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print('Teacher freed from memory.')

## Cell 8 — Diagnostic: which output index is melanoma?

Sanity check the class-index assumption. Output index 1 should have a low mean probability (~1-3%) and a small argmax count (~tens to hundreds), consistent with melanoma's rarity. Index 0 should be the dominant benign class.

In [ ]:
print(f'test_probs shape: {test_probs.shape}')
print(f'Expected: ({len(df_test)}, 9) for the 9-class model.\n')

print('Per-class statistics on the test set:')
for i in range(test_probs.shape[1]):
    mean_p = test_probs[:, i].mean()
    argmax_count = (test_probs.argmax(axis=1) == i).sum()
    marker = '  <-- MELANOMA' if i == MELANOMA_IDX else ''
    print(f'  Class {i}: mean prob = {mean_p:.4f}, argmax count = {argmax_count:>5}{marker}')

expected_mel = int(0.0176 * len(df_test))
print(f'\nExpected real melanomas in test (~1.76% base rate): ~{expected_mel}')
actual_mel_argmax = (test_probs.argmax(axis=1) == MELANOMA_IDX).sum()
print(f'Argmax count for class {MELANOMA_IDX} (melanoma): {actual_mel_argmax}')

if actual_mel_argmax > 1000:
    print('\n⚠️  WARNING: too many predicted melanomas. Either MELANOMA_IDX is wrong, ')
    print('   or the teacher model is mis-calibrated. Stop and investigate before retraining.')
elif actual_mel_argmax < 10:
    print('\n⚠️  WARNING: very few predicted melanomas. The pseudolabel signal will be weak.')
    print('   Consider lowering CONFIDENCE_THRESHOLD or using P(melanoma) directly instead of argmax.')
else:
    print('\n✓ Melanoma prediction count is in a sensible range.')

## Cell 9 — Filter pseudolabels and inspect

We keep test samples that are **confident** (max softmax above threshold). Their class distribution is the dataset we'll feed into training.

In [ ]:
from collections import Counter

confidence = test_probs.max(axis=1)
pseudo_class = test_probs.argmax(axis=1)

keep_mask = confidence >= CONFIDENCE_THRESHOLD
n_kept = keep_mask.sum()
n_discarded = (~keep_mask).sum()

print(f'Confidence threshold: {CONFIDENCE_THRESHOLD}')
print(f'Test images kept:      {n_kept} / {len(df_test)} ({100*n_kept/len(df_test):.1f}%)')
print(f'Test images discarded: {n_discarded}')

kept_classes = pseudo_class[keep_mask]
class_counts = Counter(kept_classes.tolist())

print('\nPseudolabel class distribution (kept samples):')
for cls_idx in sorted(class_counts.keys()):
    label = 'MELANOMA' if cls_idx == MELANOMA_IDX else f'class {cls_idx}'
    print(f'  Class {cls_idx} ({label}): {class_counts[cls_idx]}')

n_pseudo_melanoma = class_counts.get(MELANOMA_IDX, 0)
n_pseudo_benign = n_kept - n_pseudo_melanoma

print(f'\nPseudo-melanomas (rare class):  {n_pseudo_melanoma}')
print(f'Pseudo-benigns:                  {n_pseudo_benign}')
print(f'Real melanomas in training:      {(df_trn["target"]==1).sum()}')
print(f'After {PSEUDO_MELANOMA_UPSAMPLE}× upsampling, effective pseudo-melanomas: {n_pseudo_melanoma * PSEUDO_MELANOMA_UPSAMPLE}')

if n_pseudo_melanoma == 0:
    raise RuntimeError(
        'No pseudo-melanomas at this threshold. Either MELANOMA_IDX is wrong, '
        'or the threshold is too high. Lower CONFIDENCE_THRESHOLD and rerun.'
    )

## Cell 10 — Visualise confidence and melanoma-probability distributions

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(confidence, bins=50, color='#0D9488', edgecolor='black', alpha=0.7)
axes[0].axvline(CONFIDENCE_THRESHOLD, color='red', linestyle='--', linewidth=2,
                label=f'Threshold = {CONFIDENCE_THRESHOLD}')
axes[0].set_xlabel('Max softmax probability (confidence)')
axes[0].set_ylabel('Number of test images')
axes[0].set_title('Teacher confidence distribution')
axes[0].legend(); axes[0].grid(alpha=0.3)

mel_probs = test_probs[:, MELANOMA_IDX]
axes[1].hist(mel_probs, bins=50, color='#DC2626', edgecolor='black', alpha=0.7)
axes[1].set_xlabel(f'Predicted P(melanoma) [class {MELANOMA_IDX}]')
axes[1].set_ylabel('Number of test images')
axes[1].set_title('Melanoma probability across test set')
axes[1].set_yscale('log'); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

print(f'\nP(melanoma) > 0.5: {(mel_probs > 0.5).sum()} test images')
print(f'P(melanoma) > 0.8: {(mel_probs > 0.8).sum()} test images')
print(f'Expected real melanomas (~1.76%): ~{int(0.0176 * len(df_test))}')

## Cell 11 — Build the combined training dataset (CACHED)

Combine real + filtered-pseudo + upsampled pseudo-melanomas. Cached to Drive so we don't redo this if the session restarts.

In [ ]:
import pickle

if os.path.exists(COMBINED_DF_CACHE):
    print(f'✓ Loading cached combined dataframe from {COMBINED_DF_CACHE}')
    with open(COMBINED_DF_CACHE, 'rb') as f:
        df_combined = pickle.load(f)
    print(f'  Total samples: {len(df_combined)}')
else:
    # Build the pseudolabel dataframe (only kept test images)
    df_pseudo = df_test[keep_mask].reset_index(drop=True).copy()
    df_pseudo['pseudo_probs'] = list(test_probs[keep_mask])
    df_pseudo['target'] = kept_classes  # 9-class index (e.g. 0 for benign, 1 for melanoma)
    df_pseudo['is_pseudo'] = True
    
    # Upsample pseudo-melanomas
    if PSEUDO_MELANOMA_UPSAMPLE > 1:
        pseudo_mel = df_pseudo[df_pseudo['target'] == MELANOMA_IDX]
        pseudo_rest = df_pseudo[df_pseudo['target'] != MELANOMA_IDX]
        pseudo_mel_up = pd.concat([pseudo_mel] * PSEUDO_MELANOMA_UPSAMPLE, ignore_index=True)
        df_pseudo = pd.concat([pseudo_rest, pseudo_mel_up], ignore_index=True)
        print(f'Upsampled pseudo-melanomas: {len(pseudo_mel)} -> {len(pseudo_mel_up)}')
    
    # Prepare original training data. In your train.csv, target is already 0/1 binary,
    # which matches the model's class index convention (0=benign, 1=melanoma).
    df_trn_combined = df_trn.copy()
    df_trn_combined['is_pseudo'] = False
    df_trn_combined['pseudo_probs'] = None
    
    df_combined = pd.concat([df_trn_combined, df_pseudo], ignore_index=True)
    df_combined = df_combined.sample(frac=1, random_state=CFG.seed).reset_index(drop=True)
    
    with open(COMBINED_DF_CACHE, 'wb') as f:
        pickle.dump(df_combined, f)
    print(f'✓ Saved combined dataframe to {COMBINED_DF_CACHE}')

print(f'\nCombined training set composition:')
print(f'  Real samples:     {(~df_combined["is_pseudo"]).sum()}')
print(f'  Pseudo samples:   {df_combined["is_pseudo"].sum()}')
print(f'  Total:            {len(df_combined)}')
real_mel = ((~df_combined['is_pseudo']) & (df_combined['target'] == 1)).sum()
pseudo_mel_total = (df_combined['is_pseudo'] & (df_combined['target'] == MELANOMA_IDX)).sum()
print(f'  Real melanomas (target==1):                {real_mel}')
print(f'  Pseudo melanomas (target=={MELANOMA_IDX}, after upsample): {pseudo_mel_total}')

## Cell 12 — Pseudolabel-aware dataset class + custom collate

Dataset returns `(image, target, is_pseudo)`. Collate function harmonises hard and soft labels into one batch tensor.

In [ ]:
import cv2
import torch.nn.functional as F
from torch.utils.data import Dataset

class PseudoLabelDataset(Dataset):
    """Dataset that handles a mix of real labels and soft pseudolabels.
    
    Returns:
        image:     (C, H, W) tensor
        target:    scalar (hard label) or (9,) soft distribution (pseudo + soft)
        is_pseudo: bool
    
    Both real and pseudo class indices follow the same convention:
    target=0 means benign, target=1 means melanoma (and so on for the
    multi-class system, though real labels only ever use 0 or 1).
    """
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(row['filepath'])
        if image is None:
            raise FileNotFoundError(f"Image not found: {row['filepath']}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform:
            image = self.transform(image=image)['image']
        
        is_pseudo = bool(row['is_pseudo'])
        
        if is_pseudo and USE_SOFT_LABELS and row.get('pseudo_probs') is not None:
            target = torch.tensor(row['pseudo_probs'], dtype=torch.float32)
        else:
            target = torch.tensor(int(row['target']), dtype=torch.long)
        
        return image, target, is_pseudo


def custom_collate(batch):
    """If any target in the batch is soft, promote all to soft via one-hot."""
    images = torch.stack([b[0] for b in batch])
    targets_raw = [b[1] for b in batch]
    is_pseudo = torch.tensor([b[2] for b in batch], dtype=torch.bool)
    
    any_soft = any(t.dim() == 1 and t.numel() > 1 for t in targets_raw)
    
    if any_soft:
        targets = []
        for t in targets_raw:
            if t.dim() == 0:
                t = F.one_hot(t, num_classes=9).float()
            targets.append(t)
        targets = torch.stack(targets)
    else:
        targets = torch.stack(targets_raw)
    
    return images, targets, is_pseudo


def mixed_loss(logits, targets, is_pseudo):
    log_probs = F.log_softmax(logits, dim=1)
    if targets.dim() == 1:
        return F.nll_loss(log_probs, targets)
    if targets.dim() == 2:
        return -(targets * log_probs).sum(dim=1).mean()
    raise ValueError(f'Unexpected target shape: {targets.shape}')


# Build datasets and loaders
train_transform = get_transforms(CFG.image_size, 'train')
val_transform   = get_transforms(CFG.image_size, 'val')

train_ds = PseudoLabelDataset(df_combined, transform=train_transform)
val_ds   = MelanomaDataset(df_val, transform=val_transform)

train_loader = DataLoader(
    train_ds, batch_size=CFG.batch_size, shuffle=True,
    num_workers=CFG.num_workers, pin_memory=True, drop_last=True,
    collate_fn=custom_collate,
)
val_loader = DataLoader(
    val_ds, batch_size=CFG.batch_size * 2, shuffle=False,
    num_workers=CFG.num_workers, pin_memory=True,
)

print(f'Train batches: {len(train_loader)} ({len(train_ds)} samples)')
print(f'Val batches:   {len(val_loader)} ({len(val_ds)} samples)')

## Cell 13 — Build student + resume helpers

Helpers to save and restore the full training state (model, optimizer, scheduler, scaler, history). On a clean run this initialises fresh; on a resumed run it loads from Drive.

In [ ]:
from sklearn.metrics import roc_auc_score
import json
import time

def build_student():
    model = build_model(
        model_name=CFG.model_name,
        out_dim=9,
        pretrained=True,
        drop_rate=0.5,
    ).to(CFG.device)
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG.init_lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CFG.n_epochs, eta_min=CFG.init_lr * 0.01,
    )
    return model, optimizer, scheduler


def save_train_state(model, optimizer, scheduler, scaler, epoch, best_auc, history, path):
    state = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict() if scaler is not None else None,
        'best_auc': best_auc,
        'history': history,
    }
    torch.save(state, path)


def load_train_state(model, optimizer, scheduler, scaler, path):
    state = torch.load(path, map_location=CFG.device, weights_only=False)
    model.load_state_dict(state['model_state_dict'])
    optimizer.load_state_dict(state['optimizer_state_dict'])
    scheduler.load_state_dict(state['scheduler_state_dict'])
    if scaler is not None and state.get('scaler_state_dict') is not None:
        scaler.load_state_dict(state['scaler_state_dict'])
    return state['epoch'], state['best_auc'], state['history']


MELANOMA_OUT_IDX = MELANOMA_IDX  # for clarity in the validation function


def validate(model, loader, device):
    model.eval()
    all_targets, all_probs, val_loss_total, n = [], [], 0.0, 0
    with torch.no_grad():
        for images, targets in tqdm(loader, desc='  Valid', leave=False):
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            logits = model(images)
            val_loss_total += F.cross_entropy(logits, targets).item() * images.size(0)
            probs = torch.softmax(logits, dim=1)[:, MELANOMA_OUT_IDX]
            all_targets.append(targets.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
            n += images.size(0)
    all_targets = np.concatenate(all_targets)
    all_probs = np.concatenate(all_probs)
    val_loss = val_loss_total / n
    binary_targets = (all_targets == MELANOMA_OUT_IDX).astype(int)
    auc = roc_auc_score(binary_targets, all_probs) if 0 < binary_targets.sum() < n else 0.0
    return val_loss, auc

print('Helpers defined.')

## Cell 14 — Training loop with mid-training resume

**Key resilience feature:** the full training state is saved to Drive at the end of every epoch. If the Colab session dies mid-training, re-running this cell picks up exactly where it left off — no wasted epochs.

In [ ]:
student, optimizer, scheduler = build_student()
scaler, use_amp = make_amp_components(use_amp=CFG.use_amp, device=CFG.device)

# Resume if possible
if os.path.exists(TRAIN_STATE_CACHE):
    print(f'Resuming from {TRAIN_STATE_CACHE}')
    start_epoch_offset, best_auc, history = load_train_state(
        student, optimizer, scheduler, scaler, TRAIN_STATE_CACHE
    )
    start_epoch = start_epoch_offset + 1
    print(f'  Resumed at epoch {start_epoch}/{CFG.n_epochs} (best so far: {best_auc:.4f})')
else:
    print('Fresh training run')
    start_epoch = 1
    best_auc = 0.0
    history = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'lr': [], 'time_min': []}

print(f'\nStudent training: {EXPERIMENT_NAME}')
print('=' * 70)

for epoch in range(start_epoch, CFG.n_epochs + 1):
    lr_now = optimizer.param_groups[0]['lr']
    print(f'\nEpoch {epoch}/{CFG.n_epochs} (lr={lr_now:.2e})')
    t0 = time.time()
    
    # ---- Train ----
    student.train()
    running_loss, n_samples = 0.0, 0
    pbar = tqdm(train_loader, desc='  Train', leave=False)
    for images, targets, is_pseudo in pbar:
        images = images.to(CFG.device, non_blocking=True)
        targets = targets.to(CFG.device, non_blocking=True)
        is_pseudo = is_pseudo.to(CFG.device, non_blocking=True)
        
        optimizer.zero_grad()
        if use_amp and CFG.device.type == 'cuda':
            with torch.cuda.amp.autocast():
                logits = student(images)
                loss = mixed_loss(logits, targets, is_pseudo)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = student(images)
            loss = mixed_loss(logits, targets, is_pseudo)
            loss.backward()
            optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        n_samples += images.size(0)
        pbar.set_postfix(loss=f'{loss.item():.4f}')
    
    train_loss = running_loss / n_samples
    val_loss, val_auc = validate(student, val_loader, CFG.device)
    scheduler.step()
    elapsed = (time.time() - t0) / 60
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)
    history['lr'].append(lr_now)
    history['time_min'].append(elapsed)
    
    print(f'  Train Loss: {train_loss:.4f}')
    print(f'  Val   Loss: {val_loss:.4f}  AUC: {val_auc:.4f}')
    print(f'  Time:       {elapsed:.1f} min')
    
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save({
            'epoch': epoch,
            'model_state_dict': student.state_dict(),
            'best_auc': best_auc,
            'experiment': EXPERIMENT_NAME,
        }, f'{WEIGHTS_DIR}/{EXPERIMENT_NAME}_fold{CFG.fold}_best.pth')
        print(f'  New best AUC: {best_auc:.4f} -> best checkpoint saved')
    
    # Save resumable training state every epoch
    save_train_state(student, optimizer, scheduler, scaler, epoch, best_auc, history, TRAIN_STATE_CACHE)
    print(f'  Resume state saved -> {TRAIN_STATE_CACHE}')
    
    # Save per-epoch student weights too
    torch.save(student.state_dict(), f'{WEIGHTS_DIR}/{EXPERIMENT_NAME}_fold{CFG.fold}_epoch{epoch}.pth')
    
    # Save history JSON (compatible with compare_results.ipynb)
    with open(f'{RESULTS_DIR}/{EXPERIMENT_NAME}.json', 'w') as f:
        json.dump({
            'experiment_name': EXPERIMENT_NAME,
            'model_name': CFG.model_name,
            'image_size': CFG.image_size,
            'batch_size': CFG.batch_size,
            'fold': CFG.fold,
            'best_auc': best_auc,
            'pseudolabel_config': {
                'teacher_checkpoint': TEACHER_CHECKPOINT,
                'melanoma_idx': MELANOMA_IDX,
                'confidence_threshold': CONFIDENCE_THRESHOLD,
                'soft_labels': USE_SOFT_LABELS,
                'melanoma_upsample': PSEUDO_MELANOMA_UPSAMPLE,
                'tta_passes': TTA_PASSES if USE_TTA_FOR_PSEUDOLABELS else 0,
                'pseudo_samples_kept': int(n_kept),
                'pseudo_melanomas_before_upsample': int(n_pseudo_melanoma),
            },
            'history': history,
        }, f, indent=2)

print('\n' + '=' * 70)
print(f'Training complete!')
print(f'Best Val AUC:         {best_auc:.4f}')
print(f'Baseline (no pseudo): 0.9363')
print(f'Δ vs baseline:        {best_auc - 0.9363:+.4f}')
print('=' * 70)

## Cell 15 — Plot training history

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_x = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_x, history['train_loss'], 'o-', label='Train', color='#1B2A4A')
axes[0].plot(epochs_x, history['val_loss'], 's-', label='Val', color='#0D9488')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_x, history['val_auc'], 'o-', color='#7C3AED', linewidth=2,
             label=f'Pseudo (best {best_auc:.4f})')
axes[1].axhline(y=0.9363, color='black', linestyle='--', alpha=0.5, label='Baseline 0.9363')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val AUC')
axes[1].set_title('Validation AUC vs Baseline')
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_ylim([0.85, 0.96])

axes[2].plot(epochs_x, history['lr'], 'o-', color='#475569')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('LR')
axes[2].set_title('Learning Rate'); axes[2].set_yscale('log'); axes[2].grid(alpha=0.3)

plt.suptitle(f'{EXPERIMENT_NAME} - Pseudolabel learning', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/{EXPERIMENT_NAME}_history.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nSummary:')
print(f'  Baseline (no pseudo): 0.9363')
print(f'  Pseudolabel best:     {best_auc:.4f}')
print(f'  Δ vs baseline:        {best_auc - 0.9363:+.4f}')
print(f'  Total time:           {sum(history["time_min"]):.1f} min')

## How to recover from a session loss

If your Colab session dies, re-run the notebook from the top. Each cell will detect existing Drive artefacts and skip the expensive work:

1. **Pseudolabel generation (Cell 7):** if `test_probs.npy` exists in Drive, reload it (~5 seconds vs ~15 minutes).
2. **Combined dataframe (Cell 11):** if `df_combined.pkl` exists, reload it.
3. **Training (Cell 14):** if `train_state.pt` exists, resume from the last completed epoch with full optimizer/scheduler state.

**To force a fresh restart of any stage,** delete the relevant cache file:

```python
# To regenerate pseudolabels:
!rm -f {TEST_PROBS_CACHE}

# To rebuild combined dataframe (e.g. with different threshold):
!rm -f {COMBINED_DF_CACHE}

# To restart training from epoch 1:
!rm -f {TRAIN_STATE_CACHE}
```

## Post-experiment notes

**If pseudolabeling helped (+0.005 or more):** confirms the technique works on this dataset. Try lower thresholds, more TTA passes, or a second self-training round.

**If pseudolabeling hurt or was neutral:** likely confirmation bias. Try raising the threshold (0.9 or 0.95), disabling melanoma upsampling, or both. This would itself be an interesting finding to discuss.

**Honest reporting:** whatever the result, this is post-submission learning. The original report stands on its own merits. If you write this up later, report the result faithfully — including any negative finding.